In [103]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [105]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-6-luna'
openai = OpenAI()

API key looks good so far


In [123]:
# this package 
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [136]:
def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "no title found"
    text = ""
    # print(soup.body(["script", "style", "img", "input"]))
    if soup.body:
        for script in soup.body(["script", "style", "img", "input"]):
            script.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else: 
        text = "no body found"

    return (title + "\n\n" + text)[:2_000]    

In [108]:
def fetch_website_links(url): 
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = []
    for link in soup.find_all("a"):
        href = link.get("href")
        if href and href.startswith("http"):
            links.append(href)
    return links

In [121]:
# res =  fetch_website_contents("https://bsrm.com/")
# res = fetch_website_links("https://bsrm.com/")
# print(res)

In [109]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [143]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [137]:

def select_relevant_links(url):
    messages = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ]

    result = openai.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=messages,
        response_format={"type": "json_object"}
    )

    content = (result.choices[0].message.content)
    links = json.loads(content)
   
    urls=[]
    for link in links["links"]:
        if link["url"]:
            urls.append(link)
    return urls


In [138]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
       
    return result

In [139]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] 
    return user_prompt

In [140]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [142]:
create_brochure("Frobel Academy" , "https://www.frobelacademy.com/")

# Frobel Academy

**Joyful, purposeful learning for the whole child**

Based in Chattogram, Bangladesh, Frobel Academy aims to make high-quality education accessible in a supportive environment where children can become eager, lifelong learners.

## A child-centered approach

Frobel focuses on each learner’s academic, emotional, and social development. Its teaching is designed around children’s needs, using experiential and joyful learning to nurture critical thinking, curiosity, compassion, and confidence.

The Academy’s vision is for every child to strive for excellence as a compassionate, self-aware global learner.

## Learning inside and outside the classroom

Alongside its academic curriculum, Frobel offers activities intended to broaden students’ experience and support 21st-century skills. These include:

- **STEAMS:** Science, Technology, Engineering, Art, Mathematics, and Sports
- **Music and dance, theatre arts, and physical sports**
- **Karate, yoga, and French**
- **Community service**

## A commitment to education and community

Frobel Academy is a project of **BSRM Group of Companies**, a leading Bangladeshi steel manufacturer. BSRM’s longstanding interest in education includes support for Burhani BSRM School, which provides free education to children from slum communities. Frobel was established with the goal of offering world-class education at affordable prices. The Academy says its profits will help fund other CSR projects, especially in education and special needs.

## Contact

**Address:** Bangabandhu Avenue, Oxygen–Quaish Adjacent Road, near Ananya R/A, Wazidia, Bayazid, Chattogram-4211  
**Phone:** +880 1313-400961 · 09678372372  
**Email:** info@frobelacademy.com